# PSA Translation — NLLB-200-distilled-600M Fine-Tuning
### English/Kiswahili → Ekegusii (low-resource) machine translation

This notebook trains and evaluates **NLLB-200-distilled-600M** with layer freezing on a curated
English/Kiswahili → Ekegusii PSA (Public Service Announcement) dataset.

**No Colab, Kaggle, or Weights & Biases dependency** — designed to run on any standard
Python + GPU environment (e.g. Navon Cloud JupyterLab). All logs, metrics, and checkpoints
are written to local files under `./checkpoints/` and `./logs/`.

**Requirements:** Python 3.10+, a CUDA GPU (tested on NVIDIA T4 15GB; will run faster/larger
batches on an A100), and `Final_merged_psas.csv` placed in the same directory as this notebook
(or update `DATA_PATH` below).

**Note on memory settings below:** the batch size, gradient accumulation, gradient
checkpointing, and Adafactor optimizer choices were tuned on a 15GB T4 GPU to avoid
out-of-memory errors. On a larger GPU (e.g. an 80GB A100) these are conservative and can
likely be relaxed (e.g. larger batch size, Adam instead of Adafactor) for faster training —
but they are left as the tested, working configuration by default so this runs without
errors on any GPU size.


## 1. Setup

In [1]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate mlflow scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 23.2.0 requires cryptography!=40.0.0,!=40.0.1,<42,>=38.0.0, but you have cryptography 49.0.0 which is incompatible.


In [3]:
import os
# Restrict to a single GPU by default -- safe no-op on single-GPU machines,
# and avoids a known multi-GPU memory-overhead issue on some multi-GPU setups.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


'0'

In [4]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch

from transformers import set_seed

# Reproducibility
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Local output directories (created automatically, no cloud mounts needed)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True
Device: NVIDIA A100-SXM4-80GB


## 2. Load curated dataset

In [5]:
# Place Final_merged_psas.csv in the same folder as this notebook,
# or set DATA_PATH to its full path.
DATA_PATH = "Final_merged_psas.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


(21306, 5)


,PSA_ID,Domain,English,Kiswahili,Ekegusii
0,1,Agriculture,Farmers are urged to prioritize safe agrochemi...,Wakulima wanakumbushwa kuzingatia matumizi sal...,Abakuli batosere omogori bw'ogenda gesia bw'og...
1,2,Agriculture,Trucks ferrying top-dressing fertilizer are no...,Masafa yanayosafirisha mbolea ya kuongeza mavu...,Matika agwanana oria okora buya bwakonyang'ana...
2,3,Agriculture,Farmers in Wajir are invited to participate in...,Wakulima wa Wajir wanakaribishwa kushiriki kat...,Abagere bu Wajir batarikire kugana mu Ksh. 5 b...
3,4,Agriculture,Farmers are encouraged to participate in the s...,Wakulima wanahimizwa kushiriki katika mpango w...,Abagaba batemerewe kuhakanya mulashi wa ethano...
4,5,Agriculture,A Ksh. 34.4 billion program has been launched ...,Mpango wa Ksh. bilioni 34.4 umeanzishwa ili ku...,Programu ya Ksh. 34.4 bilioni imeanzishwa kuim...


### 2.1 Build the combined (English + Kiswahili) → Ekegusii dataset

Each row becomes **two** training examples where possible: one with English as source,
one with Kiswahili as source, both mapping to the same Ekegusii target.

In [8]:
def build_combined(df):
    rows = []
    for _, r in df.iterrows():
        if pd.notna(r["English"]) and str(r["English"]).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["English"], "target_text": r["Ekegusii"],
                "source_lang": "en"
            })
        if pd.notna(r.get("Kiswahili")) and str(r.get("Kiswahili")).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["Kiswahili"], "target_text": r["Ekegusii"],
                "source_lang": "sw"
            })
    return pd.DataFrame(rows).dropna(subset=["target_text"])

combined = build_combined(df)
combined = combined[combined["target_text"].astype(str).str.strip() != ""]
print("Total combined examples:", len(combined))
print(combined["source_lang"].value_counts())
print(combined["Domain"].value_counts())


Total combined examples: 42609
source_lang
en    21306
sw    21303
Name: count, dtype: int64
Domain
Education            10573
Agriculture           8912
Health                8176
Security & Safety     8028
Governance            6920
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import GroupShuffleSplit

# Split by PSA_ID (not by row) so the English and Kiswahili versions of the same
# PSA -- which share the identical Ekegusii target -- always land in the same split.
# Splitting on rows directly would leak target sentences across train/test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss.split(combined, groups=combined["PSA_ID"]))
train_df, temp_df = combined.iloc[train_idx].reset_index(drop=True), combined.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["PSA_ID"]))
val_df, test_df = temp_df.iloc[val_idx].reset_index(drop=True), temp_df.iloc[test_idx].reset_index(drop=True)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
print(train_df["source_lang"].value_counts())
print(test_df["source_lang"].value_counts())

# Sanity check: confirm no PSA_ID appears in more than one split
assert set(train_df["PSA_ID"]) & set(val_df["PSA_ID"]) == set()
assert set(train_df["PSA_ID"]) & set(test_df["PSA_ID"]) == set()
assert set(val_df["PSA_ID"]) & set(test_df["PSA_ID"]) == set()
print("No PSA_ID overlap between splits -- confirmed.")


Train: 34081  Val: 4256  Test: 4272
source_lang
en    17042
sw    17039
Name: count, dtype: int64
source_lang
en    2136
sw    2136
Name: count, dtype: int64
No PSA_ID overlap between splits -- confirmed.


**Low-resource note:** Ekegusii has no native language code in NLLB-200 and is not
in its training data, so this is a genuine low-resource target. A placeholder target
language tag (`swh_Latn`, Kiswahili's code) is used to steer generation, since NLLB
requires a valid target-language token at generation time.

## 3. Shared utilities: tokenization, metrics, layer freezing, timing

In [10]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def to_hf(d):
    return Dataset.from_pandas(d[["source_text", "target_text", "source_lang", "Domain"]]
                                .reset_index(drop=True))

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

def build_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        # -100 is the label-ignore sentinel; must be swapped for a real pad id before decoding
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        c = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": bleu["score"], "chrf": c["score"]}
    return compute_metrics

def freeze_encoder_layers(model, num_layers_to_freeze):
    """Freeze bottom N encoder layers to reduce overfitting risk on our small,
    low-resource fine-tuning set and cut compute cost."""
    encoder = model.get_encoder()
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers_to_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def score(preds, refs, label):
    """Computes BLEU/chrF, prints, and returns a result dict (no external logging service)."""
    bleu = sacrebleu.compute(predictions=preds, references=[[r] for r in refs])
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])
    print(f"{label:35s} BLEU={bleu['score']:.2f}  chrF={c['score']:.2f}")
    return {"name": label, "bleu": bleu["score"], "chrf": c["score"]}

MAX_LEN = 128
results_log = []          # collects every score() call for the final summary table
timing_log = {}           # collects wall-clock training time


## 4. NLLB-200-distilled-600M — tokenizer and preprocessing

In [11]:
NLLB_CHECKPOINT = "facebook/nllb-200-distilled-600M"
TGT_PLACEHOLDER = "swh_Latn"  # placeholder tag for Ekegusii (unsupported by NLLB-200)

nllb_tok = AutoTokenizer.from_pretrained(NLLB_CHECKPOINT)

def nllb_src_code(source_lang):
    return "eng_Latn" if source_lang == "en" else "swh_Latn"

def preprocess_nllb(batch):
    all_ids, all_labels = [], []
    for sl, src, tgt in zip(batch["source_lang"], batch["source_text"], batch["target_text"]):
        nllb_tok.src_lang = nllb_src_code(sl)
        nllb_tok.tgt_lang = TGT_PLACEHOLDER  # fix: force this every time, don't rely on the tokenizer's default
        enc = nllb_tok(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
        all_labels.append(enc["labels"])
    return {
        "input_ids": [e["input_ids"] for e in all_ids],
        "attention_mask": [e["attention_mask"] for e in all_ids],
        "labels": all_labels,
    }

train_tok_nllb = train_ds.map(preprocess_nllb, batched=True, batch_size=16)
val_tok_nllb   = val_ds.map(preprocess_nllb, batched=True, batch_size=16)


Map: 100%|██████████| 4256/4256 [00:01<00:00, 2454.48 examples/s]


In [12]:
import mlflow

mlflow.set_experiment("psa-translation-nllb-en-sw-to-guz")
mlflow.start_run(run_name="nllb_combined_guz")
mlflow.log_params({
    "model_checkpoint": NLLB_CHECKPOINT,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
})
# Per-epoch training/eval metrics (loss, BLEU, chrF) are logged automatically by the
# HF Trainer's built-in MLflow integration once report_to=["mlflow"] is set below --
# no manual per-step logging needed.


2026/08/11 14:26:16 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/08/11 14:26:16 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

### 4.1 Baseline (zero-shot) — NLLB, before any fine-tuning

In [13]:
base_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
base_nllb.to("cuda" if torch.cuda.is_available() else "cpu")

test_en = test_df[test_df["source_lang"] == "en"]
test_sw = test_df[test_df["source_lang"] == "sw"]

def generate_nllb(model, texts, source_langs, tgt_code=TGT_PLACEHOLDER):
    preds = []
    for sl, t in zip(source_langs, texts):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = nllb_tok.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        preds.append(nllb_tok.decode(out[0], skip_special_tokens=True))
    return preds

# NLLB inference is per-example (language-tagged), so we subsample the test set for speed.
# Increase n_eval if you have more GPU time to spare.
n_eval = min(60, len(test_en))
preds_base_en_nllb = generate_nllb(base_nllb, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_base_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_zero-shot_en-guz"))

n_eval_sw = min(60, len(test_sw))
preds_base_sw_nllb = generate_nllb(base_nllb, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_base_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_zero-shot_sw-guz"))

del base_nllb
if torch.cuda.is_available():
    torch.cuda.empty_cache()

with open("logs/nllb_results_so_far.json", "w") as f:
    json.dump(results_log, f, indent=2)


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 20135.24it/s]


nllb_zero-shot_en-guz               BLEU=5.58  chrF=22.88
nllb_zero-shot_sw-guz               BLEU=6.46  chrF=24.24


### 4.2 Fine-tuning — NLLB (few-shot), with layer freezing

Checkpoints save automatically each epoch to `checkpoints/nllb_combined_guz/`.
Training logs (loss, BLEU, chrF per epoch) are written to `logs/nllb_training_log.csv`
after training completes — no external logging service required.

Uses `Adafactor` (lower memory footprint than Adam) with gradient checkpointing and
gradient accumulation — see the note in the intro cell about relaxing these on a
larger GPU.

In [14]:
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
nllb_model = freeze_encoder_layers(nllb_model, num_layers_to_freeze=6)  # heavier model -> freeze more

args_nllb = Seq2SeqTrainingArguments(
    output_dir="checkpoints/nllb_combined_guz",
    per_device_train_batch_size=16,  # bumped from 2 -- A100 80GB has far more headroom than the 15GB T4 this was tuned for
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,  # no longer needed with a larger real batch size
    learning_rate=5e-5,           # lowered for AdamW (Adafactor needed the higher 1e-3; Adam does not)
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=20,
    report_to=["mlflow"],   # auto-logs per-epoch loss/BLEU/chrF to the active MLflow run             # no external logging service
    fp16=False,
    bf16=True,  # A100 supports bf16 natively -- similar speed to fp16 without its NaN risk
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    gradient_checkpointing=False,  # unnecessary on 80GB; was needed only on 15GB T4
    optim="adamw_torch",          # switched from adafactor -- plenty of memory for Adam on 80GB
)

trainer_nllb = Seq2SeqTrainer(
    model=nllb_model,
    args=args_nllb,
    train_dataset=train_tok_nllb,
    eval_dataset=val_tok_nllb,
    data_collator=DataCollatorForSeq2Seq(nllb_tok, model=nllb_model),
    processing_class=nllb_tok,
    compute_metrics=build_compute_metrics(nllb_tok),
)

t0 = time.time()
trainer_nllb.train()
timing_log["nllb_train_seconds"] = time.time() - t0
print(f"NLLB training time: {timing_log['nllb_train_seconds']/60:.1f} minutes")

# Save the best checkpoint (load_best_model_at_end=True already loaded it into memory) as a
# clean, inference-ready model -- this is what the demo function below loads.
trainer_nllb.save_model("checkpoints/nllb_combined_guz/final")
nllb_tok.save_pretrained("checkpoints/nllb_combined_guz/final")

# --- Save training log locally (replaces the W&B dashboard) ---
log_df = pd.DataFrame(trainer_nllb.state.log_history)
log_df.to_csv("logs/nllb_training_log.csv", index=False)
print("Training log saved to logs/nllb_training_log.csv")


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 9918.91it/s]


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,3.296146,3.255455,4.597412,30.502087
2,3.074441,3.122513,5.532312,32.373901
3,2.848479,3.076814,6.307209,33.864423
4,2.792214,3.061748,6.094973,33.798047
5,2.726014,3.060519,6.614612,34.582734


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.61s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


NLLB training time: 107.2 minutes


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.65s/it]


Training log saved to logs/nllb_training_log.csv


In [34]:
trainer_nllb.save_model("checkpoints/nllb_combined_guz/final")
nllb_tok.save_pretrained("checkpoints/nllb_combined_guz/final")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.57s/it]


('checkpoints/nllb_combined_guz/final/tokenizer_config.json',
 'checkpoints/nllb_combined_guz/final/tokenizer.json')

### 4.3 Fine-tuned evaluation — NLLB, per source language

In [15]:
preds_ft_en_nllb = generate_nllb(trainer_nllb.model, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_ft_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_few-shot_en-guz"))

preds_ft_sw_nllb = generate_nllb(trainer_nllb.model, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_ft_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_few-shot_sw-guz"))

# Persist all results (zero-shot + few-shot) locally
with open("logs/nllb_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print("Results saved to logs/nllb_results.json")


nllb_few-shot_en-guz                BLEU=3.49  chrF=25.27
nllb_few-shot_sw-guz                BLEU=3.08  chrF=25.20
Results saved to logs/nllb_results.json


### 4.4 Domain ablation — per-domain performance (fine-tuned model)

Checks whether performance holds up across PSA domains (health, agriculture, etc.),
not just in aggregate — a lightweight stand-in for full domain-adaptation analysis.


In [16]:
print("=== Domain ablation (few-shot nllb, combined en+sw) ===")
domain_results = []
for domain, group in test_df.groupby("Domain"):
    group = group.iloc[:30]  # capped for speed, same pattern as n_eval above
    preds = generate_nllb(trainer_nllb.model, list(group["source_text"]), list(group["source_lang"]))
    r = score(preds, list(group["target_text"]), f"nllb_domain_{domain}")
    r["n"] = len(group)
    domain_results.append(r)

domain_df = pd.DataFrame(domain_results)
domain_df.to_csv("logs/nllb_domain_ablation.csv", index=False)
mlflow.log_artifact("logs/nllb_domain_ablation.csv")
domain_df


=== Domain ablation (few-shot nllb, combined en+sw) ===
nllb_domain_Agriculture             BLEU=4.47  chrF=27.26
nllb_domain_Education               BLEU=1.68  chrF=26.44
nllb_domain_Governance              BLEU=12.54  chrF=40.84
nllb_domain_Health                  BLEU=9.50  chrF=38.40
nllb_domain_Security & Safety       BLEU=8.09  chrF=34.25


,name,bleu,chrf,n
0,nllb_domain_Agriculture,4.469107,27.257909,30
1,nllb_domain_Education,1.676902,26.441742,30
2,nllb_domain_Governance,12.542871,40.838979,30
3,nllb_domain_Health,9.501077,38.398743,30
4,nllb_domain_Security & Safety,8.089923,34.248199,30


### 4.5 Save full test-set predictions + confidence scores (for COMET / human eval / error analysis)

Generates translations for the **entire** test set (not the capped subsets used in the ablation
cells above) with the fine-tuned model, plus a per-sentence confidence score, and writes it all
to a CSV. This file is the raw material for next week's COMET scoring, native-speaker evaluation,
and error analysis -- none of those need the model reloaded or re-run once this exists.


In [42]:
def generate_with_confidence_nllb(model, texts, source_langs, tgt_code=TGT_PLACEHOLDER):
    """Like generate_nllb, but also returns a per-sentence confidence score
    (mean exponentiated log-probability of the generated tokens, roughly 0-1)."""
    preds, confs = [], []
    for sl, t in zip(source_langs, texts):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = nllb_tok.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                              output_scores=True, return_dict_in_generate=True)
        preds.append(nllb_tok.decode(out.sequences[0], skip_special_tokens=True))
        transition_scores = model.compute_transition_scores(
            out.sequences, out.scores, normalize_logits=True
        )
        mask = transition_scores > -1e9
        seq_logprob = (transition_scores * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        confs.append(torch.exp(seq_logprob).item())
    return preds, confs


# Full test set -- NLLB generates one example at a time (language-tagged per row), so this
# cell will take noticeably longer than the capped ablation cells above. That's expected;
# let it run to completion since this is the one file next week's tasks all depend on.
preds_nllb_full, conf_nllb_full = generate_with_confidence_nllb(
    trainer_nllb.model, list(test_df["source_text"]), list(test_df["source_lang"])
)

predictions_df = test_df[["source_text", "source_lang", "Domain", "target_text"]].copy()
predictions_df = predictions_df.rename(columns={"target_text": "reference"})
predictions_df["nllb_prediction"] = preds_nllb_full
predictions_df["nllb_confidence"] = conf_nllb_full

predictions_df.to_csv("logs/nllb_test_predictions.csv", index=False)
mlflow.log_artifact("logs/nllb_test_predictions.csv")
print(f"Saved {len(predictions_df)} test-set predictions with confidence scores "
      f"to logs/nllb_test_predictions.csv")
predictions_df.head()


Saved 4272 test-set predictions with confidence scores to logs/nllb_test_predictions.csv


,source_text,source_lang,Domain,reference,nllb_prediction,nllb_confidence
0,The 4th National Kalro Open Research Week and ...,en,Agriculture,Naki za Nane Kaliro Open Research Week na Naki...,Ong'anya wa 4 wa National Kalro Open Research ...,0.498448
1,Siku ya 4 ya Utafiti wa Taifa wa Kalro na Maon...,sw,Agriculture,Naki za Nane Kaliro Open Research Week na Naki...,Rituko 4 ria Kwaria ya Taifa ya Kalro na 20th ...,0.704811
2,Kenya is set to enhance cotton production and ...,en,Agriculture,Kenya ikirongo omokebura ibori naki omokoya. A...,Kenya ikwenda kuong'ere omokireri na omokireri...,0.409563
3,Kenya inaandaa kuboresha uzalishaji wa pamba n...,sw,Agriculture,Kenya ikirongo omokebura ibori naki omokoya. A...,Kenya ikwenda omokireri bw'ogenda omokireri na...,0.428124
4,1 million bags of subsidized fertilizers are n...,en,Agriculture,Maki 1 milioni ekeria chichieri kyabere buya b...,1 million bags of subsidised fertiliser are av...,0.729711


## 5. Hyperparameters, training time, and results

Auto-generated from what actually ran, and saved to `logs/nllb_hyperparameters.csv`
and `logs/nllb_results_table.csv` for the write-up.

In [43]:
hyperparam_table = pd.DataFrame([{
    "Model": "NLLB-200-distilled-600M",
    "Pair": "combined (en+sw)->guz",
    "Epochs": args_nllb.num_train_epochs,
    "Batch size": f"{args_nllb.per_device_train_batch_size} (x{args_nllb.gradient_accumulation_steps} accum)",
    "Learning rate": args_nllb.learning_rate,
    "Frozen encoder layers": "6/12",
    "Optimizer": args_nllb.optim,
    "fp16": args_nllb.fp16,
    "Gradient checkpointing": args_nllb.gradient_checkpointing,
    "Train time (min)": round(timing_log.get("nllb_train_seconds", 0) / 60, 1),
}])
hyperparam_table.to_csv("logs/nllb_hyperparameters.csv", index=False)
hyperparam_table


,Model,Pair,Epochs,Batch size,Learning rate,Frozen encoder layers,Optimizer,fp16,Gradient checkpointing,Train time (min)
0,NLLB-200-distilled-600M,combined (en+sw)->guz,5,16 (x1 accum),0.00005,6/12,OptimizerNames.ADAMW_TORCH,False,False,107.2


In [24]:
import re
domain_df["name"] = domain_df["name"].apply(lambda s: re.sub(r"[^A-Za-z0-9_\-. :/]", "_", str(s)))

In [22]:
results_df = pd.DataFrame(results_log)
results_df.to_csv("logs/nllb_results_table.csv", index=False)
print("=== Zero-shot vs Few-shot results ===")
print(results_df.to_string(index=False))


=== Zero-shot vs Few-shot results ===
                 name     bleu      chrf
nllb_zero-shot_en-guz 5.577203 22.883818
nllb_zero-shot_sw-guz 6.461804 24.236964
 nllb_few-shot_en-guz 3.494557 25.267779
 nllb_few-shot_sw-guz 3.081163 25.195432


In [27]:
# --- Final MLflow logging: extra hyperparameters not auto-captured, plus result artifacts ---
mlflow.log_params({
    "frozen_encoder_layers": "6/12",
    "gradient_checkpointing": args_nllb.gradient_checkpointing,
    "bf16": args_nllb.bf16,
    "fp16": args_nllb.fp16,
    "batch_size": args_nllb.per_device_train_batch_size,
    "gradient_accumulation_steps": args_nllb.gradient_accumulation_steps,
    "optimizer": args_nllb.optim,
})
mlflow.log_metrics({f"{r['name']}_bleu": r['bleu'] for r in results_log})
mlflow.log_metrics({f"{r['name']}_chrf": r['chrf'] for r in results_log})
mlflow.log_metrics({f"{row['name']}_bleu": row['bleu'] for _, row in domain_df.iterrows()})
mlflow.log_metrics({f"{row['name']}_chrf": row['chrf'] for _, row in domain_df.iterrows()})
for fpath in ["logs/nllb_hyperparameters.csv", "logs/nllb_results_table.csv", "logs/nllb_training_log.csv"]:
    mlflow.log_artifact(fpath)

mlflow.end_run()
print("MLflow run closed. Run `mlflow ui` in a terminal in this directory to view it.")


MLflow run closed. Run `mlflow ui` in a terminal in this directory to view it.


## 6. Inference demo

Loads the fine-tuned model from `checkpoints/nllb_combined_guz/final` and translates
sample sentences. Run this cell independently (after training, or in a fresh session
that has skipped straight to this section) to demonstrate translation on new input.

In [39]:
FINAL_MODEL_DIR = "checkpoints/nllb_combined_guz/final"

# If this cell is run standalone (e.g. a fresh kernel after training already happened),
# reload the fine-tuned model + tokenizer from disk instead of relying on in-memory objects.
if "trainer_nllb" not in globals():
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    nllb_tok = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
    _nllb_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR)
    _nllb_model.to("cuda" if torch.cuda.is_available() else "cpu")
else:
    _nllb_model = trainer_nllb.model

TGT_PLACEHOLDER = "swh_Latn"

def nllb_src_code(source_lang):
    return "eng_Latn" if source_lang == "en" else "swh_Latn"

def translate_psa(text, source_lang="en"):
    """
    text: input sentence (English or Kiswahili)
    source_lang: 'en' (English) or 'sw' (Kiswahili)
    Returns: Ekegusii translation from the fine-tuned NLLB model
    """
    nllb_tok.src_lang = nllb_src_code(source_lang)
    inputs = nllb_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(_nllb_model.device)
    forced_bos = nllb_tok.convert_tokens_to_ids(TGT_PLACEHOLDER)
    #out = _nllb_model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
    out = _nllb_model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                      num_beams=4, early_stopping=True,
                      no_repeat_ngram_size=3, repetition_penalty=1.3)
    return nllb_tok.decode(out[0], skip_special_tokens=True)

# --- Demo: sample PSAs ---
samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),
]

for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  NLLB ->", translate_psa(text, source_lang=lang))
    print()


[en] Farmers are urged to prioritize safe agrochemical usage this season.
  NLLB -> Abaremi basegetirwe korwa oboremi bwokwegendereria buya ase chingaki echio.

[sw] Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.
  NLLB -> Abaremi basegetirwe gukoresha amakemikari oboremi buya omwaka uno.

